<a href="https://colab.research.google.com/github/Aashutosh2021/openwakeword_tarining/blob/main/notebook/custom_human_voice_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenWakeWord: Custom Human Voice Training
This notebook is optimized for training a wake word model primarily on your **real voice recordings**.
It still generates a small amount of synthetic data to create "hard negatives" (words that sound like your wake word but aren't) to prevent false activations, but the vast majority of the training will focus on your actual voice!

### Step 1: Install Dependencies
Run this cell to install OpenWakeWord and download the required background noise datasets.

In [2]:
!git clone https://github.com/dscripka/openWakeWord.git
!git clone https://github.com/dscripka/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad
%cd openWakeWord
!pip install -e .
!pip install tensorflow
!pip install pronouncing
!pip install audiomentations
!pip install torch-audiomentations
!pip install speechbrain
!pip install mutagen
!pip install acoustics
!pip install torchinfo
!pip install torchmetrics
!pip install webrtcvad
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19
!pip install espeak-phonemizer
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

Cloning into 'openWakeWord'...
remote: Enumerating objects: 1248, done.
remote: Counting objects: 100% (724/724), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 1248 (delta 605), reused 563 (delta 563), pack-reused 524 (from 1)
Receiving objects: 100% (1248/1248), 3.23 MiB | 6.96 MiB/s, done.
Resolving deltas: 100% (776/776), done.
Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 3.86 MiB/s, done.
Resolving deltas: 100% (93/93), done.
--2026-07-25 12:44:57--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 3

### Step 2: Upload Your Voice Clips
1. Record 50-100 clips of yourself saying your wake word (16kHz, 16-bit, Mono `.wav` format).
2. Put them in a `.zip` file named `my_positive_clips.zip`.
3. Upload that `.zip` file to the Colab sidebar.
4. Run the cell below to extract them!

In [3]:
!unzip /content/my_positive_clips.zip -d /content/
print("Voice clips extracted to /content/my_positive_clips/")

Archive:  /content/my_positive_clips.zip
   creating: /content/my_positive_clips/
  inflating: /content/my_positive_clips/Recording (10).wav  
  inflating: /content/my_positive_clips/Recording (11).wav  
  inflating: /content/my_positive_clips/Recording (12).wav  
  inflating: /content/my_positive_clips/Recording (13).wav  
  inflating: /content/my_positive_clips/Recording (14).wav  
  inflating: /content/my_positive_clips/Recording (15).wav  
  inflating: /content/my_positive_clips/Recording (16).wav  
  inflating: /content/my_positive_clips/Recording (17).wav  
  inflating: /content/my_positive_clips/Recording (18).wav  
  inflating: /content/my_positive_clips/Recording (19).wav  
  inflating: /content/my_positive_clips/Recording (2).wav  
  inflating: /content/my_positive_clips/Recording (20).wav  
  inflating: /content/my_positive_clips/Recording (21).wav  
  inflating: /content/my_positive_clips/Recording (22).wav  
  inflating: /content/my_positive_clips/Recording (23).wav  
  in

### Step 3: Configure Your Wake Word
Change the `target_phrase` below to the word(s) you recorded yourself saying.

In [4]:
import yaml
import os

# Load the default config from openWakeWord to ensure no missing keys!
with open('/content/openWakeWord/examples/custom_model.yml', 'r') as f:
    config = yaml.load(f.read(), yaml.Loader)

# Override with our custom settings
config["target_phrase"] = ["ultron"] # Change this to your wake word
config["model_name"] = "custom_voice_model"
config["piper_sample_generator_path"] = "/content/openWakeWord/piper-sample-generator"
config["n_samples"] = 100 # Keep low so real voice dominates
config["n_samples_val"] = 100
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)
print("Configuration saved successfully with all default keys intact!")


Configuration saved successfully with all default keys intact!


### Step 4: Generate Synthetic Negatives & Phonetic Variations
This runs quickly because we lowered the `n_samples`. It generates words that sound similar to your wake word to teach the model what *not* to trigger on.

In [14]:
!apt-get install -y espeak-ng

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 53 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu0.1 [3,956 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libespeak-ng1 amd64 1.50+dfsg-10ubuntu0.1 [207 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 espeak-ng amd64 1.50+dfsg-1

In [18]:
import torchaudio
import os

# Get the path to torchaudio's __init__.py
torchaudio_init_path = os.path.join(os.path.dirname(torchaudio.__file__), "__init__.py")

# Append a dummy function to bypass the error
with open(torchaudio_init_path, "a") as f:
    f.write("\ndef set_audio_backend(backend):\n")
    f.write("    pass\n")

print("Torchaudio successfully patched!")

Torchaudio successfully patched!


In [20]:
import os
import scipy.io.wavfile
import numpy as np
from tqdm import tqdm
import datasets

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

print('Downloading Room Impulse Responses (for augmentation)...')
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print('Finished downloading RIRs!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [01:29,  3.03it/s]

Finished downloading RIRs!


In [22]:
import os
import scipy.io.wavfile
import numpy as np
from tqdm import tqdm
import datasets
from pathlib import Path

print('Downloading Audioset background noise...')
if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

print('Downloading FMA background noise...')
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1
for i in tqdm(range(n_hours*3600//30)):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break
print('Finished downloading background noise!')


--2026-07-25 13:09:48--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.23, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-07-25 13:09:48 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]

 99%|█████████▉| 119/120 [00:40<00:00,  2.94it/s]

Finished downloading background noise!


In [24]:
!wget -O /content/openWakeWord/piper-sample-generator/models/en-us-libritts-high.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'

--2026-07-25 13:12:26--  https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/40dc6d54-317d-4987-a114-8f2a02c04126?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-25T14%3A03%3A17Z&rscd=attachment%3B+filename%3Den-us-libritts-high.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-25T13%3A02%3A56Z&ske=2026-07-25T14%3A03%3A17Z&sks=b&skv=2018-11-09&sig=yAK57Mj63jNt5a5JuGjexTTeiL5ijWk2uGS0Iv7dpSE%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NDk4ODc0NywibmJmIjoxNzg0OTg1MTQ3LCJwYXRoIjoicmVsZWFzZ

In [25]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --generate_clips
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --augment_clips

INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading /content/openWakeWord/piper-sample-generator/models/en-us-libritts-high.pt
Traceback (most recent call last):
  File "/content/openWakeWord/openwakeword/train.py", line 676, in <module>
    generate_samples(
  File "/content/openWakeWord/piper-sample-generator/generate_samples.py", line 74, in generate_samples
    model = torch.load(model_path)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1578, in load
    raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
_pickle.UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` fr

### Step 5: Inject Your Real Human Voice Data
This cell converts your `.wav` files into OpenWakeWord features and seamlessly merges them into the training dataset.

In [ ]:
import os
import glob
import numpy as np
from openWakeWord.openwakeword.utils import AudioFeatures

custom_clips_dir = "/content/my_positive_clips"
if os.path.exists(custom_clips_dir):
    positive_clips = glob.glob(os.path.join(custom_clips_dir, "*.wav"))
    if len(positive_clips) > 0:
        print(f"Found {len(positive_clips)} custom positive clips. Extracting features...")
        F = AudioFeatures(device="cpu")
        custom_features = F.embed_clips(positive_clips, batch_size=16)

        if isinstance(custom_features, dict):
            custom_features = list(custom_features.values())[0]
        if isinstance(custom_features, list):
            custom_features = np.vstack(custom_features)

        model_name = config["model_name"]
        output_dir = os.path.join(config.get("output_dir", "my_custom_model"), model_name)
        train_path = os.path.join(output_dir, "positive_features_train.npy")
        val_path = os.path.join(output_dir, "positive_features_val.npy")

        if os.path.exists(train_path):
            existing_train = np.load(train_path)
            # We repeat the custom human features multiple times so the model focuses heavily on them.
            # MULTIPLIER: Set this to 4 (or higher) to repeat your 26 recordings multiple times.
            weight_multiplier = 4
            weighted_custom = np.tile(custom_features, (weight_multiplier, 1))
            new_train = np.vstack([existing_train, weighted_custom])
            np.save(train_path, new_train)
            print(f"Appended weighted real clips to training features. New total: {new_train.shape[0]}")

        if os.path.exists(val_path):
            existing_val = np.load(val_path)
            val_features = custom_features[:max(1, len(custom_features)//5)]
            new_val = np.vstack([existing_val, val_features])
            np.save(val_path, new_val)
            print(f"Appended real clips to validation features. New total: {new_val.shape[0]}")
    else:
        print(f"Directory {custom_clips_dir} exists, but no .wav files found.")
else:
    print(f"No custom real clips found at {custom_clips_dir}. Skipping...")

### Step 6: Train the Model!
Train the final neural network. Once complete, download the `.onnx` file from `/content/openWakeWord/my_custom_model/custom_voice_model/`.

In [ ]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --train_model